# Database Management — Build `stocks.db`
**Project:** MSFT Direction Predictor — Group 15, Block D ADS-AI BUas

**Purpose:** Merge the per-ticker CSV files produced by the data-fetch script (`data/raw/`) and load them into a single SQLite database (`db/stocks.db`). One table per ticker.

This notebook performs the **initial / full build** of the database. The *daily* rolling update (append newest trading day, drop oldest row) is handled by the separate automated fetch script, not here.

---

## Sections
1. **Connection** — open / create `stocks.db`
2. **CSV merging** — read and normalise all raw CSVs
3. **Adding / creating** — write each ticker to its own table
4. **Testing** — `SELECT *` checks and row counts
5. **CSV export** — join all tables into one wide modelling dataset
6. **Saving** — commit and close

## Tables
| Table | Columns |
|-------|---------|
| `msft_daily` | date, open, high, low, close, volume |
| `gold_prices` | date, close, volume |
| `vix_data` | date, close, volume |
| `asml_data` | date, close, volume |
| `nvda_data` | date, close, volume |
| `amd_data` | date, close, volume |
| `amzn_data` | date, close, volume |
| `crm_data` | date, close, volume |
| `pltr_data` | date, close, volume |

> **Note:** `^VIX` is an index and has no trading volume; its `volume` column will be empty/zero and is kept only for schema consistency.

## 0. Imports & Logging

In [1]:
import logging
import sqlite3
from pathlib import Path

import pandas as pd

# Configure logging once for the whole notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

logger.info("Libraries imported successfully")

2026-06-08 22:55:00,565 - INFO - Libraries imported successfully


### Configuration
Maps each raw CSV (file stem) to its destination table and the columns to keep. MSFT keeps full OHLCV; every other ticker keeps `close` + `volume` only.

In [2]:
# Project root: this notebook lives at notebooks/.../ so adjust parents
# as needed. We resolve relative to the current working directory and walk
# up until we find the 'data/raw' folder, so the path works regardless of
# how deep the notebook is nested.
def _find_project_root(marker: str = "data/raw") -> Path:
    """Walk upwards from the cwd until *marker* is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    # Fallback: assume three levels up (notebooks/<x>/<y>/)
    return here.parents[2] if len(here.parents) >= 3 else here


BASE_DIR = _find_project_root()
RAW_DIR = BASE_DIR / "data" / "raw"
DB_PATH = BASE_DIR / "db" / "stocks.db"

# file stem -> (table name, list of columns to keep besides 'date')
TICKER_CONFIG = {
    "MSFT": ("msft_daily", ["open", "high", "low", "close", "volume"]),
    "GOLD": ("gold_prices", ["close", "volume"]),
    "VIX": ("vix_data", ["close", "volume"]),
    "ASML": ("asml_data", ["close", "volume"]),
    "NVDA": ("nvda_data", ["close", "volume"]),
    "AMD": ("amd_data", ["close", "volume"]),
    "AMZN": ("amzn_data", ["close", "volume"]),
    "CRM": ("crm_data", ["close", "volume"]),
    "PLTR": ("pltr_data", ["close", "volume"]),
}

logger.info("Project root : %s", BASE_DIR)
logger.info("Raw CSV dir  : %s", RAW_DIR)
logger.info("Database     : %s", DB_PATH)

2026-06-08 22:55:00,582 - INFO - Project root : C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard
2026-06-08 22:55:00,584 - INFO - Raw CSV dir  : C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\data\raw
2026-06-08 22:55:00,585 - INFO - Database     : C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\db\stocks.db


## 1. Connection
Open a connection to `stocks.db`, creating the file (and `db/` folder) if they do not yet exist.

In [3]:
def create_connection(db_path: Path) -> sqlite3.Connection:
    """
    Open a connection to the SQLite database, creating it if needed.

    Args:
        db_path (Path): Path to the SQLite database file.

    Returns:
        sqlite3.Connection: Active database connection.
    """
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(db_path))
    logger.info("Connected to database: %s", db_path)
    return conn


conn = create_connection(DB_PATH)

2026-06-08 22:55:00,603 - INFO - Connected to database: C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\db\stocks.db


## 2. CSV merging
Each raw CSV (produced by the fetch script) has a clean single-row header with columns `date, open, high, low, close, volume`. The loader below reads a CSV, validates it, normalises the date, and keeps only the columns required for that ticker's table.

In [4]:
def load_raw_csv(csv_path: Path, keep_cols: list[str]) -> pd.DataFrame:
    """
    Read a single raw CSV and return a tidy DataFrame.

    Args:
        csv_path (Path): Path to the raw CSV file.
        keep_cols (list[str]): Columns to keep alongside 'date'
            (e.g. ['close', 'volume'] or the full OHLCV list).

    Returns:
        pd.DataFrame: Tidy frame with 'date' plus the requested columns,
        sorted ascending by date and de-duplicated.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Raw CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    # Normalise column names to lowercase for consistent access
    df.columns = [c.strip().lower() for c in df.columns]

    if "date" not in df.columns:
        raise ValueError(f"{csv_path.name}: no 'date' column found")

    # Parse and standardise the date to YYYY-MM-DD strings
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

    # Warn if any requested column is missing, then keep what we can
    missing = [c for c in keep_cols if c not in df.columns]
    if missing:
        logger.warning("%s: missing columns %s", csv_path.name, missing)
    available = [c for c in keep_cols if c in df.columns]

    df = df[["date", *available]]
    df = df.dropna(subset=["date"])
    df = df.drop_duplicates(subset="date")
    df = df.sort_values("date").reset_index(drop=True)
    return df


def merge_all_csvs(raw_dir: Path, config: dict) -> dict:
    """
    Load every configured CSV into a tidy DataFrame.

    Args:
        raw_dir (Path): Directory containing the raw CSV files.
        config (dict): Mapping of file stem -> (table, keep_cols).

    Returns:
        dict: Mapping of table name -> tidy DataFrame.
    """
    frames = {}
    for stem, (table, keep_cols) in config.items():
        csv_path = raw_dir / f"{stem}.csv"
        df = load_raw_csv(csv_path, keep_cols)
        frames[table] = df
        logger.info(
            "%-12s -> %-12s | %4d rows | %s to %s",
            stem, table, len(df),
            df["date"].min() if not df.empty else "-",
            df["date"].max() if not df.empty else "-",
        )
    return frames


frames = merge_all_csvs(RAW_DIR, TICKER_CONFIG)
print(f"Loaded {len(frames)} ticker tables from {RAW_DIR}")

2026-06-08 22:55:00,643 - INFO - MSFT         -> msft_daily   | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,664 - INFO - GOLD         -> gold_prices  | 1864 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,680 - INFO - VIX          -> vix_data     | 1863 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,693 - INFO - ASML         -> asml_data    | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,712 - INFO - NVDA         -> nvda_data    | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,732 - INFO - AMD          -> amd_data     | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,746 - INFO - AMZN         -> amzn_data    | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,760 - INFO - CRM          -> crm_data     | 1862 rows | 2019-01-02 to 2026-05-29
2026-06-08 22:55:00,772 - INFO - PLTR         -> pltr_data    | 1422 rows | 2020-09-30 to 2026-05-29


Loaded 9 ticker tables from C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\data\raw


Quick preview of the MSFT frame to confirm the schema looks correct:

In [5]:
frames["msft_daily"].head(3)

,date,open,high,low,close,volume
0,2019-01-02,92.730678,94.779972,92.162463,94.193130,35329300
1,2019-01-03,93.243025,93.326864,90.541678,90.727982,42579100
2,2019-01-04,92.889044,95.487926,92.153159,94.947655,44060600


## 3. Adding / creating the database
Write each tidy DataFrame to its own table. We use `if_exists='replace'` so this notebook always rebuilds the raw tables from scratch — the daily incremental updates are the fetch script's job, not this one's.

In [6]:
def write_tables(conn: sqlite3.Connection, frames: dict) -> None:
    """
    Write each DataFrame to its table, replacing any existing table.

    Args:
        conn (sqlite3.Connection): Active database connection.
        frames (dict): Mapping of table name -> DataFrame.
    """
    for table, df in frames.items():
        df.to_sql(table, conn, if_exists="replace", index=False)
        logger.info("Wrote %-12s (%d rows)", table, len(df))
    conn.commit()


write_tables(conn, frames)
print("All ticker tables written to the database.")

2026-06-08 22:55:00,834 - INFO - Wrote msft_daily   (1862 rows)
2026-06-08 22:55:00,850 - INFO - Wrote gold_prices  (1864 rows)
2026-06-08 22:55:00,866 - INFO - Wrote vix_data     (1863 rows)
2026-06-08 22:55:00,884 - INFO - Wrote asml_data    (1862 rows)
2026-06-08 22:55:00,900 - INFO - Wrote nvda_data    (1862 rows)
2026-06-08 22:55:00,916 - INFO - Wrote amd_data     (1862 rows)
2026-06-08 22:55:00,933 - INFO - Wrote amzn_data    (1862 rows)
2026-06-08 22:55:00,949 - INFO - Wrote crm_data     (1862 rows)
2026-06-08 22:55:00,966 - INFO - Wrote pltr_data    (1422 rows)


All ticker tables written to the database.


## 4. Testing
Confirm every table exists, list its row count, and run a sample `SELECT *` against the database to verify the data is queryable.

In [7]:
def list_tables(conn: sqlite3.Connection) -> list[str]:
    """Return the names of all tables in the database."""
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def verify_database(conn: sqlite3.Connection) -> None:
    """Print each table with its row count and a null-value check."""
    print("Tables in database:")
    for table in list_tables(conn):
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        nulls = int(df.isnull().sum().sum())
        status = "clean" if nulls == 0 else f"{nulls} missing values"
        print(f"  {table:<14} {len(df):>5} rows | {status}")


verify_database(conn)

Tables in database:
  amd_data        1862 rows | clean
  amzn_data       1862 rows | clean
  asml_data       1862 rows | clean
  crm_data        1862 rows | clean
  gold_prices     1864 rows | clean
  msft_daily      1862 rows | clean
  nvda_data       1862 rows | clean
  pltr_data       1422 rows | clean
  vix_data        1863 rows | clean


Sample `SELECT * FROM msft_daily` (most recent 5 trading days):

In [8]:
pd.read_sql(
    "SELECT * FROM msft_daily ORDER BY date DESC LIMIT 5", conn
)

,date,open,high,low,close,volume
0,2026-05-29,432.549988,450.329987,432.359985,450.239990,79654400
1,2026-05-28,412.980011,429.489990,412.670013,426.989990,47250500
2,2026-05-27,411.010010,415.940002,409.579987,412.670013,28901500
3,2026-05-26,416.429993,419.769989,413.019989,416.029999,30398000
4,2026-05-22,419.540009,424.399994,416.329987,418.570007,22390300


Sample `SELECT * FROM nvda_data` (close + volume only):

In [9]:
pd.read_sql(
    "SELECT * FROM nvda_data ORDER BY date DESC LIMIT 5", conn
)

,date,close,volume
0,2026-05-29,210.894211,289410600
1,2026-05-28,214.000580,143996000
2,2026-05-27,212.352509,167601200
3,2026-05-26,214.609879,187202600
4,2026-05-22,215.079330,169275700


## 5. Export combined modelling dataset (CSV)
Join all ticker tables into a single wide dataset — one row per date, one set of columns per ticker — and write it to a CSV for modelling.

- **Outer join** on `date`, so every date from any ticker is kept.
- Each ticker's columns are **prefixed** (e.g. `msft_close`, `nvda_volume`) to avoid collisions.
- Missing values are **forward-filled** from the last known value (then back-filled once at the very start to cover leading gaps, e.g. PLTR before its IPO).

Output: `data/processed/modelling_dataset.csv`.

In [10]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELLING_CSV = PROCESSED_DIR / "modelling_dataset.csv"


def build_modelling_dataset(conn: sqlite3.Connection, config: dict) -> pd.DataFrame:
    """
    Join every ticker table into one wide, forward-filled DataFrame.

    Args:
        conn (sqlite3.Connection): Active database connection.
        config (dict): Mapping of stem -> (table, keep_cols).

    Returns:
        pd.DataFrame: One row per date; each ticker's value columns are
        prefixed with its stem (lowercased). Sorted ascending by date.
    """
    merged = None
    for stem, (table, _keep) in config.items():
        df = pd.read_sql(f"SELECT * FROM {table}", conn)

        # Prefix every column except 'date' with the ticker stem
        prefix = stem.lower()
        df = df.rename(
            columns={
                c: f"{prefix}_{c}" for c in df.columns if c != "date"
            }
        )

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df, on="date", how="outer")

    # Sort chronologically, then forward-fill, then back-fill leading gaps
    merged = merged.sort_values("date").reset_index(drop=True)
    value_cols = [c for c in merged.columns if c != "date"]
    merged[value_cols] = merged[value_cols].ffill().bfill()

    logger.info(
        "Modelling dataset: %d rows x %d columns (%s to %s)",
        len(merged), merged.shape[1],
        merged["date"].min(), merged["date"].max(),
    )
    return merged


def export_modelling_csv(df: pd.DataFrame, csv_path: Path) -> None:
    """Write the modelling dataset to *csv_path*."""
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    logger.info("Modelling dataset saved to %s", csv_path)


modelling_df = build_modelling_dataset(conn, TICKER_CONFIG)
export_modelling_csv(modelling_df, MODELLING_CSV)
print(f"Modelling dataset: {modelling_df.shape[0]} rows x "
      f"{modelling_df.shape[1]} columns")
modelling_df.head(3)

2026-06-08 22:55:01,145 - INFO - Modelling dataset: 1865 rows x 22 columns (2019-01-02 to 2026-05-29)
2026-06-08 22:55:01,193 - INFO - Modelling dataset saved to C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\data\processed\modelling_dataset.csv


Modelling dataset: 1865 rows x 22 columns


,date,msft_open,msft_high,msft_low,msft_close,msft_volume,gold_close,gold_volume,vix_close,vix_volume,...,nvda_close,nvda_volume,amd_close,amd_volume,amzn_close,amzn_volume,crm_close,crm_volume,pltr_close,pltr_volume
0,2019-01-02,92.730678,94.779972,92.162463,94.193130,35329300.0,1281.000000,149.0,23.219999,0.0,...,3.373052,508752000.0,18.830000,87148700.0,76.956497,159662000.0,133.588730,4783900.0,9.5,338584400.0
1,2019-01-03,93.243025,93.326864,90.541678,90.727982,42579100.0,1291.800049,10.0,25.450001,0.0,...,3.169262,705552000.0,17.049999,117277600.0,75.014000,139512000.0,128.513199,6365700.0,9.5,338584400.0
2,2019-01-04,92.889044,95.487926,92.153159,94.947655,44060600.0,1282.699951,34.0,21.379999,0.0,...,3.372309,585620000.0,19.000000,111878600.0,78.769501,183652000.0,135.963837,6650600.0,9.5,338584400.0


## 6. Saving
Commit any pending changes and close the connection. The database file now lives on disk at `db/stocks.db` and is ready for the feature / model notebooks and the automated daily fetch script.

In [11]:
conn.commit()
conn.close()
logger.info("Database committed and connection closed.")
print(f"stocks.db saved at: {DB_PATH}")
print(f"Tables: {', '.join(t for t, _ in TICKER_CONFIG.values())}")

2026-06-08 22:55:01,227 - INFO - Database committed and connection closed.


stocks.db saved at: C:\Users\angel\Documents\GitHub\2025-26d-fai1-adsai-group_15\dashboard\db\stocks.db
Tables: msft_daily, gold_prices, vix_data, asml_data, nvda_data, amd_data, amzn_data, crm_data, pltr_data
